In [15]:
import pandas as pd

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f'{train_df.describe()}\n')
print(f'{train_df.info()}\n')
print(f'{train_df.isnull().sum()}\n')

                id      gas_raw       temp_c        flame      gas_avg  \
count  3232.000000  2325.000000  2322.000000  3232.000000  2340.000000   
mean   7016.380260   599.895891    70.574290     0.249691   581.776136   
std    1151.040797   406.219182    52.912426     0.432901   385.839546   
min    5000.000000  -155.204229   -21.952217     0.000000  -124.847898   
25%    6009.750000   275.473835    26.910714     0.000000   271.356831   
50%    7026.500000   600.083115    73.652402     0.000000   454.164968   
75%    8013.250000   855.381985   104.833014     0.000000   840.667084   
max    8999.000000  2172.368187   268.381581     1.000000  2010.299948   

          temp_avg         dGas        dTemp  sensor_noise  timestamp_ms  \
count  2327.000000  2263.000000  2267.000000   2324.000000  3.232000e+03   
mean     69.687802     0.822007     0.024979      0.212370  1.498010e+09   
std      52.393391    27.252869     1.595192      1.318669  2.844246e+08   
min     -19.473057  -126.9559

In [16]:
#SUBTASK 1
hazards = ['NORMAL','GAS_LEAK','OVERHEAT','FIRE']

citiri = []
med_temps = []
med_gas = []

total = len(train_df)

for h in hazards:
    subset = train_df[train_df['label'] == h]

    pct = round((len(subset) / total) * 100,2)
    citiri.append(pct)

    t_avg = round(subset['temp_c'].mean(),1)
    med_temps.append(t_avg)

    g_avg = round(subset['gas_raw'].mean(),1)
    med_gas.append(g_avg)

dpID = [x for x in range(1,13)] 

T1 = pd.DataFrame({
    'subtaskID':1,
    'datapointID':dpID,
    'answer':citiri+med_temps+med_gas
})

In [17]:
#SUBTASK 2
num_cols = ['gas_raw','temp_c','gas_avg','temp_avg','dGas','dTemp','sensor_noise',
           'battery_level','vehicle_speed','gas_sensor_drift','ambient_temp','flame']

medians = train_df[num_cols].median()

X = train_df[num_cols].fillna(medians)

label_maps = {'NORMAL':0, 'GAS_LEAK':1, 'OVERHEAT':2, 'FIRE':3}
y = train_df['label'].map(label_maps)

correlations = X.corrwith(y, method='spearman').abs()

df_corr = pd.DataFrame({
    'feature':correlations.index,
    'corr_value':correlations.values
}).sort_values(
    by=['corr_value','feature'],
    ascending=[False,True]
)

top6 = df_corr['feature'].head(6).tolist()

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X[top6])

X_test_imputed = test_df[num_cols].fillna(medians)
X_test_scaled = scaler.transform(X_test_imputed[top6])

X_scaled_df = pd.DataFrame(X_scaled, columns=top6)
normal_mask = (train_df['label'] == 'NORMAL')

X_normal = X_scaled_df[normal_mask]

In [19]:
import numpy as np
from scipy.spatial.distance import mahalanobis

mu = X_normal.mean(axis=0).values

Sigma = X_normal.cov().values

I = np.eye(Sigma.shape[0])
Sigma_reg = Sigma + I * 1e-6

Sigma_inv = np.linalg.inv(Sigma_reg)

anomaly_scores = []

for x in X_test_scaled:
    d = mahalanobis(x, mu, Sigma_inv)
    anomaly_scores.append(round(d,2))

T2 = pd.DataFrame({
    'subtaskID':2,
    'datapointID': test_df['id'],
    'answer':anomaly_scores
})

In [20]:
#SUBTASK 3
train_scores = []
for x in X_scaled:
    d = mahalanobis(x, mu, Sigma_inv)
    train_scores.append(round(d,2))

p33 = np.percentile(train_scores, 33)
p66 = np.percentile(train_scores, 66)

risk_levels = []

for score in anomaly_scores:
    if score < p33:
        risk_levels.append('LOW')
    elif score < p66:
        risk_levels.append('MEDIUM')
    else:
        risk_levels.append('HIGH')

T3 = pd.DataFrame({
    'subtaskID':3,
    'datapointID':test_df['id'],
    'answer':risk_levels
})

In [21]:
#SUBTASK 4
X_train = train_df[num_cols + ['weather_condition']].copy()
X_train[num_cols] = X_train[num_cols].fillna(medians)

X_test = test_df[num_cols + ['weather_condition']].copy()
X_test[num_cols] = X_test[num_cols].fillna(medians)

y_train = train_df['label']

cat_features = ['weather_condition']
X_train['weather_condition'] = X_train['weather_condition'].astype(str)
X_test['weather_condition'] = X_test['weather_condition'].astype(str)

from catboost import CatBoostClassifier

pisica = CatBoostClassifier(
    loss_function='MultiClass',
    cat_features=cat_features,
    random_seed=42,
    verbose=100
)
pisica.fit(X_train,y_train)
predictions = pisica.predict(X_test).ravel()

T4 = pd.DataFrame({
    'subtaskID':4,
    'datapointID':test_df['id'],
    'answer':predictions
})

Learning rate set to 0.084005
0:	learn: 1.2155657	total: 63.9ms	remaining: 1m 3s
100:	learn: 0.1119806	total: 984ms	remaining: 8.76s
200:	learn: 0.0754638	total: 1.88s	remaining: 7.49s
300:	learn: 0.0587897	total: 2.78s	remaining: 6.46s
400:	learn: 0.0482601	total: 3.67s	remaining: 5.49s
500:	learn: 0.0407889	total: 4.56s	remaining: 4.54s
600:	learn: 0.0354490	total: 5.44s	remaining: 3.61s
700:	learn: 0.0308312	total: 6.2s	remaining: 2.65s
800:	learn: 0.0270102	total: 6.79s	remaining: 1.69s
900:	learn: 0.0237611	total: 7.29s	remaining: 801ms
999:	learn: 0.0214821	total: 7.75s	remaining: 0us


In [22]:
#SUBMISSION

submission = pd.concat((T1,T2,T3,T4))
submission.columns = ['subtaskID','datapointID','answer']
submission.to_csv('submission.csv',index=False)